# Rice disease classification: executed project workflow

This notebook follows `problem and guide.pdf`. Run from the project directory with
the project Python environment. The CLI modules perform the work; saved reports
make every stage reviewable. The original cleaning notebook is preserved.

The current model study compares frozen pretrained features, then tunes the final
linear layer on the top two architectures. No growth-stage model or hardware is included.


In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
from rice_disease.data import ROOT, load_manifest

artifacts = ROOT / 'artifacts'
print('Project:', ROOT)


Project: C:\paras\college\project\Final Year Project


## 1. Dataset verification

Every image was fully decoded and checked for identical decoded pixels. Inspect
the audit below. Source images remain unchanged. To repeat the audit deliberately,
run `python -m rice_disease.data`; it refuses to overwrite a different split.


In [2]:
audit = json.loads((artifacts / 'dataset_audit.json').read_text())
display(audit)


{'dataset': 'C:\\paras\\college\\project\\Final Year Project\\Dataset\\clean_dataset-20261001T114830Z-1-001\\clean_dataset',
 'total_images': 1941,
 'class_counts': {'bacterial_leaf_blight': 297,
  'brown_spot': 320,
  'healthy': 348,
  'leaf_blast': 337,
  'leaf_scald': 308,
  'narrow_brown_spot': 331},
 'dimensions': {'1600x1600': 1939, '1600x1548': 2},
 'color_modes': {'RGB': 1941},
 'unreadable': [],
 'identical_pixel_groups': [],
 'imbalance_max_min_ratio': 1.1717171717171717,
 'segmentation_masks': 'No annotation files or mask directories found in the supplied cleaned dataset.',
 'limitations': 'Pixel hashes do not detect transformed near-duplicates or shared plant identity.'}

## 2. Train, validation and test membership

Retain the verified 389-image holdout. Stratify the original 1,552-image development
pool into 1,241 training and 311 validation images with seed 42. Validation supports
architecture selection and stopping. Test evaluation happens only after selection.


In [3]:
rows, split = load_manifest(artifacts)
display(pd.DataFrame(split['split_counts']))
print('Manifest SHA-256:', split['manifest_sha256'])


,train,validation,test
bacterial_leaf_blight,190,48,59
brown_spot,205,51,64
healthy,222,56,70
leaf_blast,215,54,68
leaf_scald,197,49,62
narrow_brown_spot,212,53,66


Manifest SHA-256: 773016ba6a13f0f92da7520a5461bda2bf70f036fc96d75c24beba52d036c026


## 3. Common preprocessing and training configuration

RGB, full-image resize to 224x224, ImageNet normalization. Original and one seeded
augmented view per training image; no augmentation for validation or test. Features
are extracted with frozen batch normalization/dropout and cached. Only the six-class
final linear layer is trained. Full-backbone fine-tuning has not been performed.


In [4]:
config = json.loads((ROOT / 'configs/benchmark.json').read_text())
display(config)


{'seed': 42,
 'models': ['mobilenet_v3_small', 'efficientnet_b0', 'resnet18', 'resnet50'],
 'weights': 'IMAGENET1K_V1',
 'image_size': 224,
 'mean': [0.485, 0.456, 0.406],
 'std': [0.229, 0.224, 0.225],
 'batch_size': 32,
 'feature_batch_size': 16,
 'workers': 0,
 'cpu_threads': 4,
 'training_views': 2,
 'epochs': 40,
 'patience': 8,
 'learning_rate': 0.001,
 'weight_decay': 0.0001,
 'optimizer': 'AdamW',
 'loss': 'CrossEntropyLoss',
 'training_mode': 'frozen_backbone_linear_probe',
 'selection_metric': 'validation_macro_f1',
 'augmentation': 'One original view and one seeded view: horizontal/vertical flips, 90-degree rotations, brightness/contrast within 10%. No crop or hue shift.'}

## 4. Four-model benchmark

The following call resumes incomplete models and preserves completed results.
First-time execution requires official weights: `python -m rice_disease.download_weights`.
Training can take several minutes on CPU. Selection uses validation macro F1.


In [5]:
from rice_disease.benchmark import run_benchmark
results = run_benchmark(ROOT / 'configs/benchmark.json', artifacts, 'frozen_baseline')
display(pd.read_csv(artifacts / 'runs/frozen_baseline/comparison.csv'))


mobilenet_v3_small: completed; preserving recorded results


efficientnet_b0: completed; preserving recorded results


resnet18: completed; preserving recorded results


resnet50: completed; preserving recorded results


,architecture,total_parameters,trainable_parameters,train_accuracy,validation_accuracy,validation_macro_f1,validation_macro_precision,validation_macro_recall,total_training_seconds,checkpoint_size_mb,inference_ms_batch1
0,resnet50,23520326,12294,0.990330,0.971061,0.971829,0.973068,0.971619,401.605737,94.390981,157.355265
1,efficientnet_b0,4015234,7686,0.983884,0.958199,0.959094,0.959971,0.958897,116.271599,16.343007,52.232640
2,resnet18,11179590,3078,0.977438,0.945338,0.945805,0.945933,0.946400,166.411544,44.794891,60.240345
3,mobilenet_v3_small,1524006,6150,0.975020,0.942122,0.943122,0.944714,0.942561,164.502866,6.221397,18.870185


## 5. Targeted head learning-rate tuning

After all four baseline models complete, compare rates 0.0003 and 0.003 on the two
highest validation-F1 architectures. All other settings remain fixed. Freeze the
winner and its checkpoint hash before using test data. Rerunning returns an existing
frozen selection, rather than starting further tuning.


In [6]:
from rice_disease.tune import tune
selection = tune(artifacts)
display({'checkpoint': selection['checkpoint'], 'validation': selection['metrics']['validation']})


Returning the existing frozen selection; no further tuning performed.


{'checkpoint': 'runs\\frozen_baseline\\resnet50.pt',
 'validation': {'loss': 0.12139516323804855,
  'accuracy': 0.9710610932475884,
  'macro_precision': 0.9730681562242384,
  'macro_recall': 0.9716193255778901,
  'macro_f1': 0.9718289118289118}}

## 6. Final held-out evaluation

This step consumes the reserved test set once for the frozen v1 configuration.
Do not tune against the resulting scores. Rerunning reads the existing report.


In [7]:
from rice_disease.evaluate import evaluate
evaluation = evaluate(artifacts)
display(evaluation['test'])
display(pd.DataFrame(evaluation['per_class']).T)


Returning existing final evaluation; no test inference repeated.


{'loss': 0.18094922602176666,
 'accuracy': 0.9434447300771208,
 'macro_precision': 0.9455699745123116,
 'macro_recall': 0.9449271619693821,
 'macro_f1': 0.9442814784862673}

,precision,recall,f1-score,support
bacterial_leaf_blight,1.000000,1.000000,1.000000,59.000000
brown_spot,0.869565,0.937500,0.902256,64.000000
healthy,0.932432,0.985714,0.958333,70.000000
leaf_blast,0.934426,0.838235,0.883721,68.000000
leaf_scald,0.953125,0.983871,0.968254,62.000000
narrow_brown_spot,0.983871,0.924242,0.953125,66.000000
accuracy,0.943445,0.943445,0.943445,0.943445
macro avg,0.945570,0.944927,0.944281,389.000000
weighted avg,0.944711,0.943445,0.943082,389.000000


## 7. Figures, application, and remaining data requirements

Run `python -m rice_disease.report` for learning curves and confusion matrices.
Launch `python -m streamlit run app.py --server.address 127.0.0.1` for the laptop app.

No pixel-level masks exist in the supplied data. Segmentation training requires
reviewed annotations for background, unaffected leaf, and lesions. See
`docs/SEGMENTATION_PLAN.md`. The app can measure a supplied annotation but cannot
predict affected area from classification scores.

Cultural-management references are recorded in `knowledge_base/diseases.json`.
Chemical rules and quantities remain disabled until current product labels,
jurisdiction, formulation, dose and application conditions are verified. See
`docs/KNOWLEDGE_BASE.md`. No dosage or treatment threshold has been invented.
